# Broken Latents: Studying SAEs and Feature Co-occurrence in Toy Models

In [A is for Absorption: Studying Feature Splitting and Absorption in Sparse Autoencoders](https://arxiv.org/abs/2409.14507) we find evidence for a phenomenon we call "Feature absorption", where a latent which seems to track a concept has arbitrary holes in its recall. We hypothesized that this is due to the sparsity penalty incentiving the SAE to partially merge features that co-occur together to increase sparsity.

In an earlier post on [Toy Models of Feature Absorption](https://www.lesswrong.com/posts/kcg58WhRxFA9hv9vN/toy-models-of-feature-absorption-in-saes), we showed that feature absorption arises when features co-occur together. We also showed that tied SAEs seemed to mitigate feature absorption in a simple toy setting.

This is a follow-up to that work, where we show that feature co-occurrence can still cause problems in tied SAEs in the realistic scenario where there are fewer SAE latents than true features.*italicized text*


## Install dependencies

In [ ]:
!pip install sae-lens rich

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.1/143.1 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.2/143.2 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.0/920.0 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 102.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.5/177.5 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 109.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.7/739.7 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━

## Controlling feature firing and co-occurrence

Below, we set up a function `get_training_batch()` which we can use to control how many ground-truth features we have, their firing probabilities and magnitudes, and an option `modify_firing_features` callback which can be used to modify the firing features in a batch, for instance forcing a feature to fire or not fire depending on other firing features.

In [ ]:
import torch
from typing import Callable

DEFAULT_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPS = 1e-8

def get_training_batch(
    batch_size: int,
    firing_probabilities: torch.Tensor, # these are the independent probabilities of each feature firing
    mean_firing_magnitudes: torch.Tensor | None = None, # If not provided, the mean firing magnitudes will all be 1
    std_firing_magnitudes: torch.Tensor | None = None, # If not provided, the stdev of magnitudes will be 0
    device: torch.device = DEFAULT_DEVICE,
    modify_firing_features: Callable[[torch.Tensor], torch.Tensor] | None = None,
):
    firing_features = torch.bernoulli(
        firing_probabilities.unsqueeze(0).expand(batch_size, -1).to(device)
    )
    if std_firing_magnitudes is None:
        std_firing_magnitudes = torch.zeros_like(firing_probabilities)
    if mean_firing_magnitudes is None:
        mean_firing_magnitudes = torch.ones_like(firing_probabilities)
    if modify_firing_features is not None:
        firing_features = modify_firing_features(firing_features)
    firing_features = firing_features.to(device)
    mean_firing_magnitudes = mean_firing_magnitudes.to(device)
    std_firing_magnitudes = std_firing_magnitudes.to(device)
    firing_magnitude_delta = torch.normal(
        torch.zeros_like(firing_probabilities).unsqueeze(0).expand(batch_size, -1).to(device),
        std_firing_magnitudes.unsqueeze(0).expand(batch_size, -1).to(device)
    )
    firing_magnitude_delta[firing_features == 0] = 0
    return (firing_features * mean_firing_magnitudes + firing_features * firing_magnitude_delta).relu()

## Creating a toy model

Our toy model is simply a linear layer which embeds the features into a hidden dimension. We initialize the model by making these feature embeddings as orthogonal as possible. If there are more features than dimensions, then these features will be in superposition. The toy model defines the "true direction" for each feature. Our goal is that a trained SAE will perfectly recover these true feature directions.

In [ ]:
import torch
from torch import nn
from transformer_lens.hook_points import HookedRootModule, HookPoint
from typing import Any
from tqdm.autonotebook import tqdm

def cos_sims(mat1: torch.Tensor, mat2: torch.Tensor):
    return (mat1 / (mat1.norm(dim=0, keepdim=True) + EPS)).T @ (mat2 / (mat2.norm(dim=0, keepdim=True) + EPS))

# based on https://github.com/3b1b/videos/blob/master/_2024/transformers/almost_orthogonal.py
def orthogonalize(num_vectors: int, vector_len: int, target_cos_sim: float = 0) -> torch.Tensor:
    "Try to make the embeddings as orthogonal as possible, putting vectors into superposition"
    embeddings = torch.randn(num_vectors, vector_len)
    embeddings /= embeddings.norm(p=2, dim=1, keepdim=True)  # Normalize
    embeddings.requires_grad_(True)
    num_vectors = embeddings.shape[0]

    # Set up an Optimization loop to create nearly-perpendicular vectors
    optimizer = torch.optim.Adam([embeddings], lr=0.01) # type: ignore
    num_steps = 1000

    losses = []

    pbar = tqdm(range(num_steps))
    for step_num in pbar:
        optimizer.zero_grad()

        dot_products = embeddings @ embeddings.T
        # Punish deviation from orthogonal
        diff = dot_products - target_cos_sim
        diff.fill_diagonal_(0)
        loss = diff.pow(2).sum()
        # Extra incentive to keep rows normalized
        loss += num_vectors * (dot_products.diag() - 1).pow(2).sum()

        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        pbar.set_description(f"loss: {loss.item():.3f}")
    embeddings = (embeddings / embeddings.norm(p=2, dim=1, keepdim=True)).detach().clone()
    embeddings.requires_grad_(False)
    return embeddings.detach().clone()


class ToyModel(HookedRootModule):
    def __init__(self, num_feats: int, hidden_dim: int, target_cos_sim: float = 0):
        super().__init__()
        self.embed = torch.nn.Linear(num_feats, hidden_dim, bias=False)
        embeddings = orthogonalize(num_feats, hidden_dim, target_cos_sim=target_cos_sim)
        self.embed.weight.data = embeddings.T
        self.setup()

    def forward(self, x: torch.Tensor, **kwargs: Any):
        x = self.embed(x)
        return x

Let's begin with a simple toy model with 4 features and an SAE with 4 latents and no superposition.

In [ ]:
small_toy_model = ToyModel(num_feats=4, hidden_dim=50).to(DEFAULT_DEVICE)

loss: 0.000: 100%|██████████| 1000/1000 [00:02<00:00, 343.04it/s]


In [ ]:
import plotly.express as px

feature_cos_sims = cos_sims(small_toy_model.embed.weight, small_toy_model.embed.weight)

px.imshow(
    feature_cos_sims.detach().cpu().numpy(),
    color_continuous_scale="RdBu",
    zmin=-1,
    zmax=1,
    title="True features cosine similarities",
    height=600,
    width=600,
)

We see that the 4 true features are all perfectly orthogonal

### Plotting helpers

Some helpers for plotting feature vs latent cosine similarities and showing features with corresponding SAE activations. You can just run these.

In [ ]:
import torch
from sae_lens.training.sae_trainer import TrainingSAE
from torch import nn
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from rich.jupyter import print as rprint
from rich.table import Table
from rich.panel import Panel
from rich.text import Text

import pandas as pd
import numpy as np


def plot_latent_firing_histograms(
        sae: TrainingSAE,
        toy_model: ToyModel,
        activations_batch_generator: Callable[[int], torch.Tensor],
        num_sample_acts: int = 100_000,
        firing_threshold: float = 0.5,
    ):
    latent_acts = sae.encode(toy_model(activations_batch_generator(100_000)))
    B, D = latent_acts.shape

    # Calculate grid dimensions
    n_cols = min(5, D)
    n_rows = math.ceil(D / n_cols)

    # Create subplot grid
    fig = make_subplots(rows=n_rows, cols=n_cols,
                       subplot_titles=[f'Latent {i}' for i in range(D)])

    # Create histograms
    for i in range(D):
        row = i // n_cols + 1
        col = i % n_cols + 1

        values = latent_acts[:, i].detach().cpu().float().numpy()
        values = values[values > firing_threshold]
        fig.add_trace(
            go.Histogram(
                x=values,
                nbinsx=25,
                xbins=dict(
                    start=0,
                    end=values.max(),
                    size=(values.max())/30
                ),
            ),
            row=row, col=col,

        )
        fig.update_xaxes(range=[0, values.max()], row=row, col=col, title_text="Firing magnitude")
        if col == 1:
            fig.update_yaxes(row=row, col=col, title_text="Count")

    suffix = f"({num_sample_acts} sample activations)" if n_cols > 1 else ""
    fig.update_layout(
        height=300 * n_rows,
        width=300 * n_cols,
        showlegend=False,
        title_text=f"SAE Latent firing distribution {suffix}"
    )
    fig.show()


def plot_latent_cos_sims(sae: TrainingSAE):
    latent_cos_sims = cos_sims(sae.W_dec.T, sae.W_dec.T)
    px.imshow(
        latent_cos_sims.detach().cpu().numpy(),
        color_continuous_scale="RdBu",
        zmin=-1,
        zmax=1,
        title="SAE latent cosine similarities",
        height=400,
        width=400,
    ).show()

def plot_training_vals(vals: list[torch.Tensor] | list[int], title: str, log_y: bool = False):
    # Convert tensor to numpy for easier handling
    data_np = torch.stack([torch.tensor(v) for v in vals]).numpy()
    if data_np.ndim == 1:
        data_np = data_np.reshape(-1, 1)
    d = data_np.shape[1]
    b = data_np.shape[0]
    t = torch.linspace(0, 1, b).unsqueeze(1)

    # Create a DataFrame
    df = pd.DataFrame(data_np, columns=[f'dim_{i}' for i in range(d)])
    df['time'] = t.squeeze().numpy()

    # Melt the DataFrame to long format
    df_melted = df.melt(id_vars=['time'], var_name='dimension', value_name='value')

    # Create the plot
    fig = px.line(df_melted, x='time', y='value', color='dimension',
                title=title,
                labels={'value': 'Value', 'time': 'Time'},
                line_dash='dimension',
                log_y=log_y,
    )

    # Update layout for better visibility
    fig.update_layout(
        xaxis_title='Time',
        yaxis_title='Value',
        legend_title='Dimension',
        hovermode="x unified"
    )

    # Show the plot
    fig.show()

def plot_tied_sae_feat_cos_sims(
    sae: TrainingSAE,
    model: ToyModel,
    subtitle: str,
    height: int = 600,
):
    enc_cos_sims = cos_sims(sae.W_enc, model.embed.weight)

    fig = make_subplots(rows=1, cols=1, subplot_titles=(subtitle,))
    hovertemplate = "True feature: %{x}<br>SAE Latent: %{y}<br>Cosine Similarity: %{z:.3f}<extra></extra>"

    fig.add_trace(
        go.Heatmap(
            z=enc_cos_sims.detach().cpu().numpy(),
            zmin=-1,
            zmax=1,
            colorscale="RdBu",
            hovertemplate=hovertemplate,
        ),
        row=1, col=1
    )


    fig.update_layout(
        height=height,
        width=600,
        title_text="Cosine Similarity between SAE Latents and True Features",
    )
    fig.update_xaxes(title_text="True feature", row=1, col=1)
    fig.update_xaxes(title_text="True feature", row=1, col=2)
    fig.update_yaxes(title_text="SAE Latent", row=1, col=1)
    fig.update_yaxes(title_text="SAE Latent", row=1, col=2)
    fig.show()

def plot_sae_feat_cos_sims(
    sae: TrainingSAE,
    model: ToyModel,
    title_suffix: str,
    height: int = 600,
):
    dec_cos_sims = cos_sims(sae.W_dec.T, model.embed.weight)
    enc_cos_sims = cos_sims(sae.W_enc, model.embed.weight)

    fig = make_subplots(rows=1, cols=2, subplot_titles=("SAE encoder", "SAE decoder"))
    hovertemplate = "True feature: %{x}<br>SAE Latent: %{y}<br>Cosine Similarity: %{z:.3f}<extra></extra>"

    fig.add_trace(
        go.Heatmap(
            z=enc_cos_sims.detach().cpu().numpy(),
            zmin=-1,
            zmax=1,
            colorscale="RdBu",
            showscale=False,
            hovertemplate=hovertemplate,
        ),
        row=1, col=1
    )

    # Add decoder plot
    fig.add_trace(
        go.Heatmap(
            z=dec_cos_sims.detach().cpu().numpy(),
            zmin=-1,
            zmax=1,
            colorscale="RdBu",
            colorbar=dict(title="cos sim", x=1.0),
            hovertemplate=hovertemplate,
        ),
        row=1, col=2
    )

    fig.update_layout(
        height=height,
        width=1200,
        title_text=f"Cosine Similarity with True Features ({title_suffix})",
    )
    fig.update_xaxes(title_text="True feature", row=1, col=1)
    fig.update_xaxes(title_text="True feature", row=1, col=2)
    fig.update_yaxes(title_text="SAE Latent", row=1, col=1)
    fig.update_yaxes(title_text="SAE Latent", row=1, col=2)
    fig.show()

def print_sample_feats_and_acts(feats: torch.Tensor, sae: TrainingSAE, model: ToyModel, include_l1: bool = True):
    feat_mags = feats.float().to(DEFAULT_DEVICE)
    latent_acts = sae.encode(model(feats.float().to(DEFAULT_DEVICE)))

    table = Table(title="Sample feature values and corresponding SAE activations")

    # Add columns
    table.add_column("True features", justify="center")
    table.add_column("SAE Latent acts", justify="center")
    if include_l1:
        table.add_column("L1", justify="center")

    def style_row(row):
        text = Text()
        for val in row:
            style = "bold" if val > 1e-4 else "dim"
            text.append(f"{val:.2f}", style=style)
            text.append("  ")
        return text

    # Add rows
    for row1, row2 in zip(feat_mags, latent_acts):
        text = Text()
        row = [style_row(row1), style_row(row2)]
        if include_l1:
            row.append(style_row(row2.sum().unsqueeze(0)))
        table.add_row(*row)
    rprint(table)

## SAE training

Here, we set up a training loop to train a SAE using SAELens on our toy features and activations. This is a slightly hacky version of the main `sae_training_runner` in SAELens. We set up the SAE and the toy model to both have the same number of features.

We study untied and tied SAE throughout this notebook, and create the TiedSAE class here as well.

In [ ]:
import wandb
import torch
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from torch.nn import init
import math
from tqdm import tqdm

from sae_lens import LanguageModelSAERunnerConfig
from sae_lens.training.sae_trainer import TrainingSAE, SAETrainer
from sae_lens.training.training_sae import TrainingSAEConfig
from sae_lens.training.geometric_median import compute_geometric_median

class TiedSAE(TrainingSAE):
    "Hackily tie the W_enc and W_dec together, and remove b_enc"

    def __init__(self, cfg: TrainingSAEConfig):
        # this is not necessary for tied SAEs
        cfg.scale_sparsity_penalty_by_decoder_norm = False
        super().__init__(cfg)

    @property
    def W_enc(self):
        return self.W_dec.T

    @property
    def b_enc(self):
        return 0

    def __setattr__(self, name, value):
        if name == "W_enc" or name == "b_enc":
            return
        super().__setattr__(name, value)

    def fold_W_dec_norm(self):
        pass


# We only need a way to get `next_batch()` from the ActivationsStore, but the real
# ActivationsStore expects tokens and a LLM rather than toy activations. For our
# purposes, this just implements the important interface to use our feature generator
class FakeActivationsStore:
    def __init__(self, model, generate_batch_fn, batch_size: int):
        self.model = model
        self.batch_size = batch_size
        self.generate_batch_fn = generate_batch_fn
        self.estimated_norm_scaling_factor = None

    def set_norm_scaling_factor_if_needed(self):
        pass

    @torch.no_grad()
    def next_batch(self):
        # the middle param is always 1 in SAELens, I think for legacy reasons
        return self.model(self.generate_batch_fn(self.batch_size)).unsqueeze(1)

# Ignore saving checkpoints, the toy models train very fast
def _save_checkpoint(trainer: SAETrainer, checkpoint_name: int | str, wandb_aliases: list[str] | None = None):
    pass

def train_toy_sae(
    d_sae: int,
    toy_model: ToyModel,
    activations_batch_provider: Callable[[int], torch.Tensor],
    l1: float = 5e-3,
    custom_init_fn: Callable[[TrainingSAEConfig], TrainingSAE] | None = None,
    training_tokens: int = 200_000_000,
) -> TrainingSAE:
    tqdm._instances.clear() # type: ignore

    cfg = LanguageModelSAERunnerConfig(
        context_size=500,
        d_in=toy_model.embed.weight.shape[0],
        d_sae=d_sae,
        device=str(DEFAULT_DEVICE),
        training_tokens=training_tokens,
        eval_every_n_wandb_logs=99999999999,
        l1_coefficient=l1,
        normalize_sae_decoder=False,
        normalize_activations="none",
        init_encoder_as_decoder_transpose=True,
        scale_sparsity_penalty_by_decoder_norm=True,
        lr=3e-4,
        log_to_wandb=False,
        apply_b_dec_to_input=True,
    )
    assert cfg.d_sae is not None
    toy_model.eval()
    sae_cfg = TrainingSAEConfig.from_dict(cfg.get_training_sae_cfg_dict())
    if custom_init_fn is not None:
        sae = custom_init_fn(sae_cfg)
    else:
        sae = TrainingSAE(sae_cfg)
    store = FakeActivationsStore(toy_model, activations_batch_provider, sae.cfg.context_size)

    trainer = SAETrainer(
        model=toy_model,
        sae=sae,
        activation_store=store, # type: ignore
        cfg=cfg,
        save_checkpoint_fn=_save_checkpoint,
    )
    trainer.fit()
    sae.fold_W_dec_norm()
    return sae

## Training a tied SAE on fully independent features

Let's start by training a SAE to recover 4 features, all fully independent of each other. The SAE should be able to do this near perfectly.

In [ ]:
from sae_lens import LanguageModelSAERunnerConfig
from functools import partial

feat_probs = torch.tensor([0.2, 0.2, 0.2, 0.2])
generate_batch = partial(get_training_batch, firing_probabilities=feat_probs)

sae_tied_small = train_toy_sae(d_sae=4, toy_model=small_toy_model, activations_batch_provider=generate_batch, custom_init_fn=TiedSAE)

48800| l1_loss: 0.00210 | mse_loss: 0.00000: 100%|█████████▉| 199884800/200000000 [01:27<00:00, 2273190.88it/s]


In [ ]:
plot_tied_sae_feat_cos_sims(sae_tied_small, small_toy_model, "Fully independent features")

The SAE is able to basically perfectly recover the feature directions here. The encoder correctly filters out each feature, and the decoder perfectly matches the true feature directions. The encoder learns slightly offset directions to allow it to filter out the interference due to overlapping features.

In [ ]:
test_feats = torch.tensor([
    [1, 0, 0, 0],
    [1, 1, 0, 0],
    [0, 0, 1, 0],
    [1, 1, 1, 1],
])

print_sample_feats_and_acts(test_feats, sae_tied_small, small_toy_model)

    Sample feature values and corresponding SAE activations     
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃      True features       ┃     SAE Latent acts      ┃   L1   ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│  1.00  0.00  0.00  0.00  │  0.00  1.00  0.00  0.00  │  1.00  │
│  1.00  1.00  0.00  0.00  │  0.00  1.00  1.00  0.00  │  2.00  │
│  0.00  0.00  1.00  0.00  │  1.00  0.00  0.00  0.00  │  1.00  │
│  1.00  1.00  1.00  1.00  │  1.00  1.00  1.00  1.00  │  3.99  │
└──────────────────────────┴──────────────────────────┴────────┘

## Tied SAEs solves absorption when the number of latents equals the number of features

Next, we'll reproduce our finding from the previous post that tied SAEs are resistant to absorption in this simple setting where the number of features and the number of SAE latents are the same.

In this setting, feature 0 fires any time feature 1 or feature 2 fires.

In [ ]:
from sae_lens import LanguageModelSAERunnerConfig
from functools import partial

feat_probs = torch.tensor([0.2, 0.2, 0.2, 0.2])

def modify_feats(feats: torch.Tensor):
    feat_1_fires = feats[:, 1] == 1
    feat_2_fires = feats[:, 2] == 1
    feats[feat_1_fires, 0] = 1
    feats[feat_2_fires, 0] = 1
    return feats

generate_batch = partial(
    get_training_batch,
    firing_probabilities=feat_probs,
    modify_firing_features=modify_feats,
)
sae_tied_small_absorb = train_toy_sae(4, small_toy_model, generate_batch, custom_init_fn=TiedSAE)

48800| l1_loss: 0.00292 | mse_loss: 0.00000: 100%|█████████▉| 199884800/200000000 [01:29<00:00, 2245272.29it/s]


In [ ]:
plot_tied_sae_feat_cos_sims(sae_tied_small_absorb, small_toy_model, "Co-occurrence: feat 0 fires if feat 1 or feat 2 fires")

In [ ]:
test_feats = torch.tensor([
    [1, 0, 0, 0],
    [1, 1, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, 1],
])

print_sample_feats_and_acts(test_feats, sae_tied_small_absorb, small_toy_model)

    Sample feature values and corresponding SAE activations     
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃      True features       ┃     SAE Latent acts      ┃   L1   ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│  1.00  0.00  0.00  0.00  │  0.00  0.00  0.71  0.71  │  1.41  │
│  1.00  1.00  0.00  0.00  │  0.00  0.00  0.00  1.41  │  1.41  │
│  0.00  0.00  1.00  0.00  │  1.00  0.00  0.00  0.00  │  1.00  │
│  0.00  0.00  0.00  1.00  │  0.00  1.00  0.00  0.00  │  1.00  │
└──────────────────────────┴──────────────────────────┴────────┘

The tied SAE still learns a perfect representation of the underlying features despite co-occurrence.

# More SAE latents than features

Previously, we've looked at what happens when the number of latents in the SAE match the number of true features, but what happens if the SAE has more latents than there are true features? Will the SAE still learn correct representations and simply kill off excess latents, as we'd hope, or will it use that extra capacity to find a way to cheat by learning combinations of latents?

First, let's try using 4 true features, all independent, but an 8-latent untied SAE.

In [ ]:
from sae_lens import LanguageModelSAERunnerConfig
from functools import partial

from functools import partial

feat_probs = torch.tensor([0.2, 0.2, 0.2, 0.2])

generate_batch = partial(
    get_training_batch,
    firing_probabilities=feat_probs,
)
# this seems to learn different things every time it's run. Try a few runs to get a sense for the resulting SAE.
sae_untied_excess_width = train_toy_sae(8, small_toy_model, generate_batch, training_tokens=300_000_000)

73200| l1_loss: 0.00407 | mse_loss: 0.00001: 100%|█████████▉| 299827200/300000000 [02:27<00:00, 2038210.44it/s]


In [ ]:
plot_sae_feat_cos_sims(sae_untied_excess_width, small_toy_model, "Overly Wide untied SAE, independent features")

In [ ]:
test_feats = torch.tensor([
    [1, 0, 0, 0],
    [1, 1, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, 1],
    [1, 1, 1, 1],
    [0, 1, 0, 1],
])

print_sample_feats_and_acts(test_feats, sae_untied_excess_width, small_toy_model)

                Sample feature values and corresponding SAE activations                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃      True features       ┃                 SAE Latent acts                  ┃   L1   ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│  1.00  0.00  0.00  0.00  │  1.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  │  1.00  │
│  1.00  1.00  0.00  0.00  │  0.00  0.00  0.00  1.41  0.00  0.00  0.00  0.00  │  1.41  │
│  0.00  0.00  1.00  0.00  │  0.00  0.00  1.00  0.00  0.00  0.00  0.00  0.00  │  1.00  │
│  0.00  0.00  0.00  1.00  │  0.00  0.00  0.00  0.00  0.00  1.00  0.00  0.00  │  1.00  │
│  1.00  1.00  1.00  1.00  │  0.00  0.00  1.00  1.41  0.00  0.99  0.00  0.00  │  3.40  │
│  0.00  1.00  0.00  1.00  │  0.00  0.00  0.00  0.00  0.00  1.00  0.63  0.37  │  1.99  │
└──────────────────────────┴──────────────────────────────────────────────────┴────────┘

The untied SAE learns some correct decoder representations of true features, but also includes some mixtures of features, and some dead latents. For several feature combinations, the L1 is lower than the L1 of true features, so the SAE seems to have misused some of its extra representational capacity to find degenerate solutions that increase sparsity.

Next, let's try the same setup, but with a tied SAE

In [ ]:
from sae_lens import LanguageModelSAERunnerConfig
from functools import partial

from functools import partial

feat_probs = torch.tensor([0.2, 0.2, 0.2, 0.2])

generate_batch = partial(
    get_training_batch,
    firing_probabilities=feat_probs,
)
# this sometimes ends up in bad local minima. If that happens, try rerunning the cell and hopefully it should work.
sae_tied_excess_width = train_toy_sae(8, small_toy_model, generate_batch, training_tokens=300_000_000, custom_init_fn=TiedSAE)

73200| l1_loss: 0.00397 | mse_loss: 0.00000: 100%|█████████▉| 299827200/300000000 [02:14<00:00, 2226273.49it/s]


In [ ]:
plot_tied_sae_feat_cos_sims(sae_tied_excess_width, small_toy_model, "Overly Wide Tied SAE, independent features")

In [ ]:
test_feats = torch.tensor([
    [1, 0, 0, 0],
    [1, 1, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, 1],
    [1, 1, 1, 1],
    [0, 1, 0, 1],
])

print_sample_feats_and_acts(test_feats, sae_tied_excess_width, small_toy_model)

                Sample feature values and corresponding SAE activations                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃      True features       ┃                 SAE Latent acts                  ┃   L1   ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│  1.00  0.00  0.00  0.00  │  0.00  0.00  0.00  0.00  0.00  0.00  1.00  0.00  │  1.00  │
│  1.00  1.00  0.00  0.00  │  0.00  0.00  0.00  1.00  0.00  0.00  1.00  0.00  │  2.00  │
│  0.00  0.00  1.00  0.00  │  1.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  │  1.00  │
│  0.00  0.00  0.00  1.00  │  0.00  0.00  0.00  0.00  1.00  0.00  0.00  0.00  │  1.00  │
│  1.00  1.00  1.00  1.00  │  1.00  0.00  0.00  1.00  1.00  0.00  1.00  0.00  │  3.98  │
│  0.00  1.00  0.00  1.00  │  0.00  0.00  0.00  1.00  1.00  0.00  0.00  0.00  │  2.00  │
└──────────────────────────┴──────────────────────────────────────────────────┴────────┘

The tied SAE correctly learns a single latent per true feature, and kills off the excess latents. This is exactly what we hope should happen.

# Tied SAEs with too many latents are still resistant to feature absorption

Now, let's add feature co-occurence, which would trigger absorption in untied SAEs. We'll use the same setup as before, where feature 0 must fire if either feature 1 or feature 2 fires. Since we know untied SAEs can't handle absorption, we'll just use a tied SAE here.

In [ ]:
from sae_lens import LanguageModelSAERunnerConfig
from functools import partial

from functools import partial

feat_probs = torch.tensor([0.2, 0.2, 0.2, 0.2])

def modify_feats(feats: torch.Tensor):
    feat_1_fires = feats[:, 1] == 1
    feat_2_fires = feats[:, 2] == 1
    feats[feat_1_fires, 0] = 1
    feats[feat_2_fires, 0] = 1
    return feats

generate_batch = partial(
    get_training_batch,
    firing_probabilities=feat_probs,
    modify_firing_features=modify_feats,
)
sae_tied_small_absorb_excess_width = train_toy_sae(8, small_toy_model, generate_batch, custom_init_fn=TiedSAE, training_tokens=500_000_000)

122000| l1_loss: 0.00302 | mse_loss: 0.00001: 100%|█████████▉| 499712000/500000000 [03:51<00:00, 2159984.21it/s]


In [ ]:
plot_tied_sae_feat_cos_sims(sae_tied_small_absorb_excess_width, small_toy_model, "Overly Wide Tied SAE, feat 0 fires if feat 1 or 2 fires")

Excellent! The SAE learns the 4 correct features despite feature co-occurrence, avoiding absorption, and still kills off its extra 4 latents.

# More features than SAE latents

In reality, we don't know how many true features there are, and our SAEs almost certainly have fewer latents than there are true features. This means it is not possible for the SAE to achieve 0 MSE loss, and must pick which latents to learn and which to skip.

## How does the SAE decide which latents to represent?

SAEs try to minimize MSE loss, which has a penalty on the square magnitude of the reconstruction error. As such, we expect that the SAE will choose to represent features that fire frequently and with high magnitude.

We explore this by creating a model with 20 features but a SAE with only 5 latents. The features have firing frequencies ranging from 0 to 0.3, and with magnitudes from 20.0 down to 1.0. We plot this below.

In [ ]:
import torch
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from tqdm import tqdm

tqdm._instances.clear() # type: ignore

feat_probs = 0.3 * (torch.arange(20) + 1) / 20
feat_magnitudes = 20 - torch.arange(20)

large_toy_model = ToyModel(num_feats=20, hidden_dim=50).to(DEFAULT_DEVICE)



fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=(
        "Feature firing probabilities",
        "Feature magnitudes",
        "Feature probability × magnitude²"
    )
)

fig.add_trace(
    go.Bar(
        x=list(range(len(feat_probs))),
        y=feat_probs.tolist(),
        name="Firing probability"
    ),
    row=1, col=1
)

fig.add_trace(
    go.Bar(
        x=list(range(len(feat_magnitudes))),
        y=feat_magnitudes.tolist(),
        name="Firing magnitude"
    ),
    row=1, col=2
)

combined = feat_probs * (feat_magnitudes ** 2)
fig.add_trace(
    go.Bar(
        x=list(range(len(combined))),
        y=combined.tolist(),
        name="Prob × magnitude²"
    ),
    row=1, col=3
)

# Update layout
fig.update_layout(
    height=400,
    width=1200,  # Adjust this value based on your display needs
    showlegend=False,
    xaxis_title="Feature number",
    xaxis2_title="Feature number",
    xaxis3_title="Feature number",
    yaxis_title="Firing probability",
    yaxis2_title="Firing magnitude",
    yaxis3_title="Prob × magnitude²"
)

fig.show()

loss: 0.003: 100%|██████████| 1000/1000 [00:04<00:00, 242.09it/s]


We expect that the SAE should choose to represent features 4, 5, 6, 7, and 8 as these have the largest probability times squared magnitude. Representing these 5 features should thus minimize MSE loss.

First, let's try this with an untied SAE.

In [ ]:
from sae_lens import LanguageModelSAERunnerConfig
from functools import partial

generate_batch = partial(
    get_training_batch,
    firing_probabilities=feat_probs,
    mean_firing_magnitudes=feat_magnitudes,
)
sae_untied_large = train_toy_sae(5, large_toy_model, generate_batch, l1=5e-2)

48800| l1_loss: 0.37229 | mse_loss: 127.95395: 100%|█████████▉| 199884800/200000000 [01:43<00:00, 1932844.86it/s]


In [ ]:
plot_sae_feat_cos_sims(sae_untied_large, large_toy_model, "Untied SAE, Independent features")

As expected, the untied SAE represents the top features by expected MSE when all features are independent. Next we'll check with a tied SAE.

In [ ]:
from sae_lens import LanguageModelSAERunnerConfig
from functools import partial

generate_batch = partial(
    get_training_batch,
    firing_probabilities=feat_probs,
    mean_firing_magnitudes=feat_magnitudes,
)
sae_tied_large = train_toy_sae(5, large_toy_model, generate_batch, custom_init_fn=TiedSAE, l1=5e-2)

48800| l1_loss: 0.33888 | mse_loss: 114.70659: 100%|█████████▉| 199884800/200000000 [01:33<00:00, 2136819.21it/s]


In [ ]:
plot_tied_sae_feat_cos_sims(sae_tied_large, large_toy_model, "Tied SAE, Independent features")

We see that the SAE has indeed learned the same 5 latents we expected it should.

# Co-occurrence with non-represented features

What happens if some of the features that the SAE explicitly represents also co-occurs with a feature that is not represented?

Next, we modify the above scenario so that feature 12 must fire when features 5 or 7 fire.

In [ ]:
import torch
from sae_lens import LanguageModelSAERunnerConfig
from functools import partial

def modify_feats(feats: torch.Tensor):
    feat_5_fires = feats[:, 5] == 1
    feat_7_fires = feats[:, 7] == 1
    feats[feat_5_fires, 12] = 1
    feats[feat_7_fires, 12] = 1
    return feats

generate_batch = partial(
    get_training_batch,
    firing_probabilities=feat_probs,
    modify_firing_features=modify_feats,
    mean_firing_magnitudes=feat_magnitudes,
)
sae_tied_large_absorb_1 = train_toy_sae(5, large_toy_model, generate_batch, custom_init_fn=TiedSAE, l1=5e-2)

48800| l1_loss: 0.45832 | mse_loss: 122.78778: 100%|█████████▉| 199884800/200000000 [01:35<00:00, 2103655.00it/s]


In [ ]:
plot_tied_sae_feat_cos_sims(sae_tied_large_absorb_1, large_toy_model, "Feat 12 fires if feat 5 or 7 fires")

Here, feature 12 is absorbed into the latents tracking features 5 and 7. Worse, the SAE has learned a negative firing pattern between features 5 and 7, since if they both fire at the same time then too much of the feature 12 direction will be added to the residual stream. The result is that the SAE latent tracking feature 5 has a negative component on feature 7. Clearly, this is not what we want! Our SAE latents are polluted with combinations of multiple features, including the latent tracking feature 5 having a negative component of feature 7.

In [ ]:
plot_latent_firing_histograms(sae_tied_large_absorb_1, large_toy_model, generate_batch)

For the latents that track features without any co-occurrence, we see them firing at only 1 magnitude. Howevever, for the latents that track features with absorption, we see them firing at multiple magnitudes depending on which combination of absorbing features fire. Maybe we can use this observation to combat absorption in real SAEs where we don't know the underlying true features?

In [ ]:
plot_latent_cos_sims(sae_tied_large_absorb_1)

The SAE latents are all orthogonal to each other, so a standard orthogonality loss won't do anything to help here.

# Can we solve this by exploiting further assumptions about the ground-truth features?

In the above scenario, the absorbed (general) feature, feature 12, is not explicitly reprented in the SAE. Feature 12 fires with magnitude 8, while the represented features fire with magnitudes greater than 11. Thus, the latents absorbing feature 12 will have two major modes:

- When feature 12 fires alone, latent magntude will be < 8.
- When feature 12 and a co-occuring feature fires, latent magnitude will be > 10.

### Latents with absortion will likely have multiple "modes" depending on which underlying co-occurring features are firing together.

Can we detect these modes and add a dynamic orthogonality loss with a threshold tailored to each latent? Below, we track the maximum firing threshold of each latent, and set a jumprelu threshold to at 70% of that value. Then, we add an orthogonality loss to encourage each latent to be orthogonal to activations which do not cause it to fire.

We implement these ideas in our `PartialOrthoTiedSAE` class below.

In [ ]:
import torch
from torch.nn.functional import normalize

class MovingMaxTracker:
    """
    Tracks the maximum value for each index over a sliding window of batches.
    Uses a circular buffer approach to avoid storing all batches.
    """

    @torch.no_grad()
    def __init__(
        self,
        hidden_dim: int,
        window_size_batches: int,
        device: torch.device = DEFAULT_DEVICE,
    ):
        self.hidden_dim = hidden_dim
        self.window_size_batches = window_size_batches
        self.device = device
        # Circular buffer to store values
        self.buffer = torch.zeros((window_size_batches, hidden_dim), device=device)
        self.running_max = torch.zeros(hidden_dim, device=device)
        self.current_pos = 0
        self.is_filled = False

    @torch.no_grad()
    def update(self, batch_hidden: torch.Tensor) -> torch.Tensor:
        # Get maximum values across batch dimension
        batch_max = torch.max(batch_hidden, dim=0)[0]
        # Store in circular buffer
        self.buffer[self.current_pos] = batch_max
        self.current_pos = (self.current_pos + 1) % self.window_size_batches
        if self.current_pos == 0:
            self.is_filled = True
        if self.is_filled:
            self.running_max = torch.max(self.buffer, dim=0)[0]
        else:
            self.running_max = torch.max(self.buffer[: self.current_pos + 1], dim=0)[0]
        return self.running_max


class PartialOrthoTiedSAE(TiedSAE):
    def __init__(
        self,
        cfg: TrainingSAEConfig,
        threshold_level: float = 0.7,
        ortho_coefficient: float = 1e4,
        add_ortho_loss_after_n_steps: int = 12_000,
        max_magnitudes_window_size_batches: int = 50,
        ortho_loss_warm_up_steps: int = 12_000,
        cos_sim_threshold: float = 0.0,
        # jumprelu activation for better performance, but still works without it
        use_jumprelu_act: bool = False
    ):
        super().__init__(cfg)
        assert cfg.architecture == "standard"
        self.step_num = 0
        self.threshold_level = threshold_level
        self.cos_sim_threshold = cos_sim_threshold
        self.ortho_coefficient = ortho_coefficient
        self.ortho_loss_warm_up_steps = ortho_loss_warm_up_steps
        self.add_ortho_loss_after_n_steps = add_ortho_loss_after_n_steps
        self.use_jumprelu_act = use_jumprelu_act
        self.max_magnitudes_tracker = MovingMaxTracker(
            self.cfg.d_sae,
            window_size_batches=max_magnitudes_window_size_batches,
            device=self.W_dec.device,
        )

    @property
    def warmup_scale(self):
        if self.step_num <= self.add_ortho_loss_after_n_steps:
            return 0
        warmup_step_num = self.step_num - self.add_ortho_loss_after_n_steps
        return min(warmup_step_num / self.ortho_loss_warm_up_steps, 1.0)

    @property
    def threshold(self):
        base_threshold = self.max_magnitudes_tracker.running_max * self.threshold_level
        threshold = self.warmup_scale * base_threshold
        threshold.requires_grad = False
        return self.warmup_scale * base_threshold

    def encode_with_hidden_pre(self, x: torch.Tensor):
        feats, hidden_pre = super().encode_with_hidden_pre(x)
        if self.use_jumprelu_act:
            feats = feats * (feats > self.threshold)
        return feats, hidden_pre

    def training_forward_pass(
        self,
        sae_in: torch.Tensor,
        current_l1_coefficient: float,
        dead_neuron_mask: torch.Tensor | None = None,
    ):
        base_output = super().training_forward_pass(
            sae_in, current_l1_coefficient, dead_neuron_mask
        )
        self.max_magnitudes_tracker.update(base_output.feature_acts)

        if self.step_num > self.add_ortho_loss_after_n_steps:
            feat_acts = base_output.feature_acts
            act_dec_cos = cos_sims(self.process_sae_in(sae_in).T, self.W_dec.T)

            act_dec_cos[feat_acts > self.threshold] = 0
            non_firing_ortho_loss = (
                self.ortho_coefficient
                * self.warmup_scale
                * ((act_dec_cos.abs() - self.cos_sim_threshold).relu() ** 2).mean()
            )

            base_output.losses["non_firing_ortho_loss"] = non_firing_ortho_loss
            base_output.loss = base_output.loss + non_firing_ortho_loss

        self.step_num += 1
        return base_output


In [ ]:
from sae_lens import LanguageModelSAERunnerConfig
from functools import partial

def modify_feats(feats: torch.Tensor):
    feat_5_fires = feats[:, 5] == 1
    feat_7_fires = feats[:, 7] == 1
    feats[feat_5_fires, 12] = 1
    feats[feat_7_fires, 12] = 1
    return feats

generate_batch = partial(
    get_training_batch,
    firing_probabilities=feat_probs,
    modify_firing_features=modify_feats,
    mean_firing_magnitudes=feat_magnitudes,
)
sae_tied_large_absorb_ortho = train_toy_sae(
    d_sae=5,
    toy_model=large_toy_model,
    activations_batch_provider=generate_batch,
    custom_init_fn=PartialOrthoTiedSAE,
    training_tokens=150_000_000,
)

36600| l1_loss: 0.03646 | mse_loss: 133.26218 | non_firing_ortho_loss: 0.53256: 100%|█████████▉| 149913600/150000000 [01:41<00:00, 1474443.44it/s]


In [ ]:
plot_tied_sae_feat_cos_sims(sae_tied_large_absorb_ortho, large_toy_model, "Feat 12 fires if feat 5 or 7 fires, ortho loss")

This works! The SAE now learns only the true latents again, without absorption.

# What if the co-occurring feature is tracked by the SAE?

Next, we adjust the setup above so that feature 6, which we saw is tracked by the SAE, must fire if features 5 or 7 fire.

In [ ]:
from sae_lens import LanguageModelSAERunnerConfig
from functools import partial

def modify_feats(feats: torch.Tensor):
    feat_5_fires = feats[:, 5] == 1
    feat_7_fires = feats[:, 7] == 1
    feats[feat_5_fires, 6] = 1
    feats[feat_7_fires, 6] = 1
    return feats

generate_batch = partial(
    get_training_batch,
    firing_probabilities=feat_probs,
    modify_firing_features=modify_feats,
    mean_firing_magnitudes=feat_magnitudes,
)
sae_tied_large_absorb_2 = train_toy_sae(5, large_toy_model, generate_batch, custom_init_fn=TiedSAE)

Run name: 5-L1-0.005-LR-0.0003-Tokens-1.000e+08
n_tokens_per_buffer (millions): 6.4
Lower bound: n_contexts_per_buffer (millions): 0.00064
Total training steps: 24414
Total wandb updates: 2441
n_tokens_per_feature_sampling_window (millions): 81920.0
n_tokens_per_dead_feature_window (millions): 40960.0
We will reset the sparsity calculation 12 times.
Number tokens in sparsity calculation window: 8.19e+06


24400| l1_loss: 0.05351 | mse_loss: 109.61052: 100%|█████████▉| 99942400/100000000 [01:27<00:00, 1136178.30it/s]


In [ ]:
plot_tied_sae_feat_cos_sims(sae_tied_large_absorb_2, large_toy_model, "Feat 6 fires if feat 5 or 7 fires")

This is even worse! The absorption patterns here are even stronger than previously. Interestingly, the SAE no longer represents feature 6 directly, but now represents feature 3.

*It looks like the SAE treats every co-occurrence pattern as its own feature*. So, while feature 6 has the largest `prob * mag^2` value on its own (0.105 * 14^2 = 20.58), it appears that the SAE views this instead as the following:


| Co-occurrence | prob  |
| ------------- | ----- |
| feats 5 and 6 | 0.090 |
| feats 7 and 6 | 0.120 |
| feat 6 alone  | 0.084 |

Now, the probability of feature 6 firing on its own without any co-occurrence (0.084) is not high enough for it to be represented in the top 5 latents by expected MSE, so the SAE represents it only in absorption!


In [ ]:
plot_latent_firing_histograms(sae_tied_large_absorb_2, large_toy_model, generate_batch)

In [ ]:
plot_latent_cos_sims(sae_tied_large_absorb_2)

## Does our partial ortho loss SAE solve this too?

In [ ]:
from sae_lens import LanguageModelSAERunnerConfig
from functools import partial

def modify_feats(feats: torch.Tensor):
    feat_5_fires = feats[:, 5] == 1
    feat_7_fires = feats[:, 7] == 1
    feats[feat_5_fires, 6] = 1
    feats[feat_7_fires, 6] = 1
    return feats

generate_batch = partial(
    get_training_batch,
    firing_probabilities=feat_probs,
    modify_firing_features=modify_feats,
    mean_firing_magnitudes=feat_magnitudes,
)
sae_tied_large_absorb_ortho_2 = train_toy_sae(5, large_toy_model, generate_batch, custom_init_fn=PartialOrthoTiedSAE, training_tokens=150_000_000)

36600| l1_loss: 0.04013 | mse_loss: 141.17363 | non_firing_ortho_loss: 2.92730: 100%|█████████▉| 149913600/150000000 [02:12<00:00, 1129257.90it/s]


In [ ]:
plot_tied_sae_feat_cos_sims(sae_tied_large_absorb_ortho_2, large_toy_model, "Feat 6 fires if feat 5 or 7 fires")

In [ ]:
plot_latent_firing_histograms(sae_tied_large_absorb_ortho_2, large_toy_model, generate_batch, firing_threshold=1.0)

# Co-occurrence with Superposition

Thus far, we've been working with fully orthogonal features. In reality, we expect that features will overlap each other slightly. Will our solution work to solve absorption when there's also superposition?

Below, we create a toy model with 20 features in 19 dimensions. We'll keep the same feature firing magnitudes and probabilities as before so everything is comparable, only with superposition this time.

In [ ]:
super_toy_model = ToyModel(num_feats=20, hidden_dim=19).to(DEFAULT_DEVICE)

loss: 1.050: 100%|██████████| 1000/1000 [00:03<00:00, 300.89it/s]


In [ ]:
import plotly.express as px

feature_cos_sims = cos_sims(super_toy_model.embed.weight, super_toy_model.embed.weight)

px.imshow(
    feature_cos_sims.detach().cpu().numpy(),
    color_continuous_scale="RdBu",
    zmin=-1,
    zmax=1,
    title="True features cosine similarities",
    height=600,
    width=600,
)

We can see that the features all overlap slightly, with a cosine similarity of ±0.05 to each other feature.

Next, we'll train a standard tied SAE with co-occurrence between features 5 and 7 and feature 6. If features 5 or 7 fire, feature 6 must also fire.

In [ ]:
from sae_lens import LanguageModelSAERunnerConfig
from functools import partial

def modify_feats(feats: torch.Tensor):
    feat_5_fires = feats[:, 5] == 1
    feat_7_fires = feats[:, 7] == 1
    feats[feat_5_fires, 6] = 1
    feats[feat_7_fires, 6] = 1
    return feats

generate_batch = partial(
    get_training_batch,
    firing_probabilities=feat_probs,
    modify_firing_features=modify_feats,
    mean_firing_magnitudes=feat_magnitudes,
)
super_sae_tied_large_absorb = train_toy_sae(
    5,
    super_toy_model,
    generate_batch,
    training_tokens=150_000_000,
    l1=1e-2
    custom_init_fn=TiedSAE,
)

36600| l1_loss: 0.25566 | mse_loss: 108.87520: 100%|█████████▉| 149913600/150000000 [02:19<00:00, 1070830.05it/s]


In [ ]:
plot_tied_sae_feat_cos_sims(super_sae_tied_large_absorb, super_toy_model, "Feat 6 fires if feat 5 or 7 fires, with superposition")

This looks similar to the non-superposition case, where the SAE learns clean representations for features 3,4, and 8, but messed-up latents for features 5,6 and 7. Next, we'll check if our modified tied SAE training method will learn correct features despite the co-occurrence pattern.

In [ ]:
from sae_lens import LanguageModelSAERunnerConfig
from functools import partial

def modify_feats(feats: torch.Tensor):
    feat_5_fires = feats[:, 5] == 1
    feat_7_fires = feats[:, 7] == 1
    feats[feat_5_fires, 6] = 1
    feats[feat_7_fires, 6] = 1
    return feats

generate_batch = partial(
    get_training_batch,
    firing_probabilities=feat_probs,
    modify_firing_features=modify_feats,
    mean_firing_magnitudes=feat_magnitudes,
)
super_sae_tied_large_absorb_ortho = train_toy_sae(
    5,
    super_toy_model,
    generate_batch,
    custom_init_fn=lambda cfg: PartialOrthoTiedSAE(
        cfg,
        ortho_coefficient=1e4,
        cos_sim_threshold=0.02,
        add_ortho_loss_after_n_steps=18_000,
        threshold_level=0.6,
    ),
    training_tokens=150_000_000,
    l1=1e-2,
)

36600| l1_loss: 0.09052 | mse_loss: 140.65808 | non_firing_ortho_loss: 7.66694: 100%|█████████▉| 149913600/150000000 [01:56<00:00, 1290384.11it/s]


In [ ]:
plot_tied_sae_feat_cos_sims(super_sae_tied_large_absorb_ortho, super_toy_model, "Feat 6 fires if feat 5 or 7 fires, with superposition")

While this isn't a perfect reconstruction, it's quite close, with cosine sim of around 0.98 to each true feature, and only very minor signs of absorption. The SAE is not representing feature 6, though, despite it being the feature with the largest expected MSE.

In [ ]:
plot_latent_cos_sims(super_sae_tied_large_absorb_ortho)

# Extreme Narrowness: Matryoshka SAEs

So far our experiments with narrow SAEs still have the SAE needing to represent both the parent feature and the child features in the same SAE. What if we make the SAE so narrow that only the parent feature can be represented? Surely, such an SAE would perfectly reconstruct the parent feature without any interference from child features?

This is the idea behind Matryoshka SAEs from [Noa Nabeshima](https://www.lesswrong.com/posts/zbebxYCqsryPALh8C/matryoshka-sparse-autoencoders#fn-gTupTd3B3GQegCM2j-4) and [Bart Bussman](https://www.lesswrong.com/posts/rKM9b6B2LqwSB5ToN/learning-multi-level-features-with-matryoshka-saes). In a Matryoshka SAE, the SAE needs to reconstruct the input using subsets of latents of increasing size. This allow the narrower SAE sizes to represent parent features, hopefully without any feature absorption, and then latents in the larger nesting size of the Matryoshka SAE can perfectly represent child features.

## Co-occurrence breaks single-latent SAEs

We test the hypothesis that a narrow SAE will perfectly learn parent features by training a 1-latent SAE in a toy setting with 4 latents in a parent-child relationship. In our toy model, feature 0 is the parent feature, and features 1 and 2 are child features. Feature 3 fires independently. Feature 0 fires with probability 0.3, and features 1 and 2 both fire with probability 0.4 if feature 0 is active. Feature 3 fires with probabily 0.2. All features fire with magnitude 1.0.

We begin by training a single-latent untied SAE on this setup. We hope this SAE's single latent will perfectly represent our parent feature, feature 0.

In [ ]:
from sae_lens import LanguageModelSAERunnerConfig
from functools import partial

feat_probs = torch.tensor([0.3, 0.4, 0.4, 0.2])

def modify_feats(feats: torch.Tensor):
    feat_0_does_not_fire = feats[:, 0] == 0
    feats[feat_0_does_not_fire, 1] = 0
    feats[feat_0_does_not_fire, 2] = 0
    return feats

generate_batch = partial(
    get_training_batch,
    firing_probabilities=feat_probs,
    modify_firing_features=modify_feats,
)

sae_untied_ultra_small = train_toy_sae(d_sae=1, toy_model=small_toy_model, activations_batch_provider=generate_batch)

48800| l1_loss: 0.00155 | mse_loss: 0.28749: 100%|█████████▉| 199884800/200000000 [01:23<00:00, 2398074.94it/s]


In [ ]:
plot_sae_feat_cos_sims(sae_untied_ultra_small, small_toy_model, "single-latent untied SAE, feat 0 fires if feats 1 or 2 fire", height=300)

We see the SAE does mainly represent feature 0 in its single latent, but it also merges in the child features 1 and 2. Feature 3, the independent feature, is fully excluded. Interestingly, the encoder of the untied SAE is nearly identical to the decoder, so the pattern is indeed different than our original absorption pattern for untied SAEs where the encoder for a parent feature had a negative rather than positive cos sim with child features. While this is not technically absorption, this is a broken latent.

Next, let's try training a tied SAE.

In [ ]:
from sae_lens import LanguageModelSAERunnerConfig
from functools import partial

feat_probs = torch.tensor([0.3, 0.4, 0.4, 0.2])

def modify_feats(feats: torch.Tensor):
    feat_0_does_not_fire = feats[:, 0] == 0
    feats[feat_0_does_not_fire, 1] = 0
    feats[feat_0_does_not_fire, 2] = 0
    return feats

generate_batch = partial(
    get_training_batch,
    firing_probabilities=feat_probs,
    modify_firing_features=modify_feats,
)

sae_tied_ultra_small = train_toy_sae(d_sae=1, toy_model=small_toy_model, activations_batch_provider=generate_batch, custom_init_fn=TiedSAE)

48800| l1_loss: 0.00185 | mse_loss: 0.29085: 100%|█████████▉| 199884800/200000000 [01:21<00:00, 2440768.29it/s]


In [ ]:
plot_tied_sae_feat_cos_sims(sae_tied_ultra_small, small_toy_model, "single-latent tied SAE, feat 0 fires if feats 1 or 2 fire", height=300)

In [ ]:
plot_latent_firing_histograms(sae_tied_ultra_small, small_toy_model, generate_batch, firing_threshold=0.0)